### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

- A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)

- A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so why do parrots talk? Hmm, I need to break this down. First, I know that parrots are known for their ability to mimic human speech, but why do they do that? Let me start by thinking about their natural behavior. Parrots are social animals, right? They live in flocks, so maybe they learn communication from each other. In the wild, they might use vocalizations to interact with their flock members, find mates, or warn others of danger.\n\nBut when it comes to talking like humans, that\'s different. Maybe it\'s a learned behavior. I remember reading that parrots have a specialized vocal organ called the syrinx, which allows them to produce a wide range of sounds. Unlike humans, who use their vocal cords, the syrinx is located where the trachea splits into the lungs, so they can make complex sounds. So their anatomy supports mimicry.\n\nBut why do they specifically mimic human speech? Maybe it\'s because they\'re in close contact with humans. If a parrot 

In [5]:
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [7]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. So I\'ll call get_weather with location set to "Boston". Make sure the JSON is correctly formatted.\n', 'tool_calls': [{'id': 'yw4tjxqen', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 154, 'total_tokens': 239, 'completion_time': 0.123014065, 'completion_tokens_details': {'reasoning_tokens': 61}, 'prompt_time': 0.006256568, 'prompt_tokens_details': None, 'queue_time': 0.159427241, 'total_time': 0.129270633}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e5bda-a0ac-74a0-98e0-699

### Tool Execution Loops

In [9]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny. A perfect day to enjoy outdoor activities! 🌞


In [10]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'r24yyz6hc', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 153, 'total_tokens': 247, 'completion_time': 0.140256892, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.006326733, 'prompt_tokens_details': None, 'queue_time': 0.055999326, 'total_time': 0.146583625}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finis